In [10]:
import csv
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit import IndicProcessor

CKPT = "ai4bharat/indictrans2-indic-en-dist-200M"
SRC_LANG = "san_Deva"
TGT_LANG = "eng_Latn"
MAX_LEN = 256
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

ip = IndicProcessor(inference=True)
tokenizer = AutoTokenizer.from_pretrained(CKPT, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(CKPT, trust_remote_code=True).to(DEVICE)
model.eval()
print("Model loaded.")

Device: cuda
Model loaded.


In [11]:
sentences = [
    "बालकः विद्यालयं गच्छति।",
    "सा पुस्तकं पठति।",
    "अहं जलं पिबामि।",
    "वृक्षे पक्षिणः वसन्ति।",
    "राजा नगरं पालयति।",
]

batch = ip.preprocess_batch(sentences, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
inputs = tokenizer(batch, padding="longest", truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)

with torch.inference_mode():
    generated = model.generate(**inputs, max_length=MAX_LEN, num_beams=5)

decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
translations = ip.postprocess_batch(decoded, lang=TGT_LANG)

for src, out in zip(sentences, translations):
    print(f"{src}\n  → {out}\n")

बालकः विद्यालयं गच्छति।
  → The child goes to school.

सा पुस्तकं पठति।
  → She reads books.

अहं जलं पिबामि।
  → I will drink the water.

वृक्षे पक्षिणः वसन्ति।
  → Birds live in trees.

राजा नगरं पालयति।
  → The king guards the city.



In [17]:
import os
os.makedirs("../outputs", exist_ok=True)

with open("../outputs/test_out.csv", "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["sanskrit", "model_output", "reference_english", "notes"])
    for src, out in zip(sentences, translations):
        writer.writerow([src, out, "", ""])
print("Saved to ../outputs/test_out.csv")

Saved to ../outputs/test_out.csv
